# Warm-up 1. How Kaggle works, and your first trained model

This is the first of two warm-ups. They come before Lesson 0. The goal is to learn
the Kaggle workflow on an easy task, so that in the knee lessons you fight only the
hard data, not the platform as well.

You use the **Titanic** getting-started competition. The task is small: predict
which passengers survived, from a table of passenger data. You will load the data,
train a model, check it, write a `submission.csv`, and see how a submission works.

**Where you are: Warm-up 1 of 2.** The full order is Warm-up 1 (this notebook),
Warm-up 2 (images on the GPU), then Lesson 0, 1, and 2 on the knee MRI data.

How to read this notebook:

- **Step** headers tell you what you do now.
- **Experiment** cells are optional. Change one setting, run the cell again, and
  read the new result.
- **Go deeper** links point to the documentation.

## Step 1: What a Kaggle notebook is

A Kaggle notebook (also called a kernel) runs on Kaggle servers, not on your
computer. You do not install anything. The main actions:

- **Join a competition and add its data.** Open the
  [Titanic competition](https://www.kaggle.com/competitions/titanic), select
  *Join*, then in a notebook use *Add Input* and select the competition. The data
  appears under `/kaggle/input/`.
- **Run a cell.** Select a cell and press Shift+Enter, or use the Run button. The
  output appears under the cell.
- **GPU.** The settings panel has an accelerator switch. This warm-up does not need
  a GPU. Warm-up 2 does.
- **Save a version.** *Save Version* runs the whole notebook top to bottom on the
  server and stores the result, including any file you write to `/kaggle/working/`.

Go deeper: [Kaggle notebooks documentation](https://www.kaggle.com/docs/notebooks).

In [ ]:
import os, glob, urllib.request
import numpy as np
import pandas as pd

# On Kaggle the competition mounts under /kaggle/input. The exact folder can vary
# (for example /kaggle/input/titanic or /kaggle/input/competitions/titanic), so we
# search /kaggle/input for the file. Set TITANIC_ROOT to run off Kaggle.
DATA_ROOT = os.environ.get("TITANIC_ROOT", "/kaggle/input")
SEED = 0

def _find(name):
    p = os.path.join(DATA_ROOT, name)
    if os.path.isfile(p):
        return p
    hits = sorted(glob.glob(os.path.join(DATA_ROOT, "**", name), recursive=True))
    return hits[0] if hits else None

TRAIN_CSV = _find("train.csv")
TEST_CSV = _find("test.csv")

# Off Kaggle (for example Colab) there is no /kaggle/input, so download the public
# Titanic CSVs. The Titanic data is public, so this is fine.
if TRAIN_CSV is None or TEST_CSV is None:
    base = "https://raw.githubusercontent.com/gsaluncf/publicfiles/main/knee-mri-kaggle/data/titanic/"
    os.makedirs("titanic_data", exist_ok=True)
    for name in ("train.csv", "test.csv"):
        dest = os.path.join("titanic_data", name)
        if not os.path.isfile(dest):
            urllib.request.urlretrieve(base + name, dest)
    TRAIN_CSV = os.path.join("titanic_data", "train.csv")
    TEST_CSV = os.path.join("titanic_data", "test.csv")
    print("Not on Kaggle: downloaded the public Titanic CSVs.")

print("train:", TRAIN_CSV)
print("test :", TEST_CSV)

## Step 2: Load the data and look at it

Read `train.csv` and look at the shape, the columns, and the survival rate. Always
look at the data before you model it.

In [ ]:
train = pd.read_csv(TRAIN_CSV)
print("rows, columns:", train.shape)
print("survival rate: %.3f" % train["Survived"].mean())
train.head()

## Step 3: Build simple features

A model needs numbers. Turn the useful columns into numbers: keep the numeric
columns, encode `Sex` as 0 or 1, and fill missing `Age` and `Fare` with the median.
This is a plain, honest starting feature set.

In [ ]:
def make_features(df):
    X = pd.DataFrame(index=df.index)
    X["Pclass"] = df["Pclass"].fillna(df["Pclass"].median())
    X["Sex_male"] = (df["Sex"] == "male").astype(int)
    X["Age"] = df["Age"].fillna(df["Age"].median())
    X["Fare"] = df["Fare"].fillna(df["Fare"].median())
    X["SibSp"] = df["SibSp"].fillna(0)
    X["Parch"] = df["Parch"].fillna(0)
    return X

X = make_features(train)
y = train["Survived"].to_numpy()
print("feature columns:", list(X.columns))
X.head()

## Step 4: Train and cross-validate

Train a `LogisticRegression` and score it with 5-fold cross-validation. The Titanic
competition uses **accuracy**, the fraction of passengers you label correctly. A
model that always predicts "did not survive" would score about 0.62 here, so beat
that.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
acc = cross_val_score(clf, X, y, cv=cv, scoring="accuracy")
print("baseline (always predict 0): %.3f" % (1 - y.mean()))
print("CV accuracy: %.3f  (per fold: %s)" % (acc.mean(), np.round(acc, 3)))

## Step 5: The shuffle control

You saw this idea named here so that it is familiar in the knee lessons. Permute the
labels at random and run the same cross-validation. With no real relationship, the
model can only predict the majority class, so the accuracy drops to the base rate.
The gap between the real score and the shuffle score is the signal your model found.

In [ ]:
rng = np.random.default_rng(SEED)
y_shuffled = y[rng.permutation(len(y))]
acc_shuffle = cross_val_score(clf, X, y_shuffled, cv=cv, scoring="accuracy")
print("real    CV accuracy: %.3f" % acc.mean())
print("shuffle CV accuracy: %.3f" % acc_shuffle.mean())

## Experiment: does a different model score higher?

Swap the logistic regression for a random forest and compare. Predict the result
before you run it. A more flexible model is not always better on a small, simple
table.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=SEED)
acc_rf = cross_val_score(rf, X, y, cv=cv, scoring="accuracy")
print("logistic regression CV accuracy: %.3f" % acc.mean())
print("random forest       CV accuracy: %.3f" % acc_rf.mean())

## Step 6: Write a submission

A Kaggle submission is a CSV with the required columns. For Titanic that is
`PassengerId` and `Survived`. Fit the model on all the training data, predict the
test passengers, and write `submission.csv` to the working directory.

To submit it: run *Save Version* (this runs the whole notebook and stores
`submission.csv`), then open the competition and submit that file, or use the
*Submit* button on the notebook output. You do not have to submit now. Writing the
file correctly is the skill for this warm-up.

In [ ]:
test = pd.read_csv(TEST_CSV)
clf.fit(X, y)
X_test = make_features(test)
predictions = clf.predict(X_test).astype(int)

submission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": predictions})
submission.to_csv("submission.csv", index=False)
print("wrote submission.csv with shape", submission.shape)
submission.head()

## Recap

You loaded competition data, built features, trained a model, checked it with
cross-validation and the shuffle control, and wrote a submission file. That is the
full Kaggle loop. Next, in Warm-up 2, you train a small model on images with the
GPU, which is the bridge to the knee MRI data in the lessons.